# SASrec

In [1]:
import os
import json
import math
import glob
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [2]:
# config
@dataclass
class Config:
    DOMAIN: str = "books"
    DATA_PATH: str = None

    USER_COL: str = "user_id"
    ITEM_COL: str = "item_id"
    TIME_COL: str = "timestamp"

    # sequential preprocessing
    DEDUP_USER_ITEM: bool = True
    DEDUP_KEEP: str = "first"
    MIN_USER_UNIQUE_ITEMS: int = 5

    # model
    MAX_SEQ_LEN: int = 50
    HIDDEN_DIM: int = 64
    NUM_HEADS: int = 2
    NUM_LAYERS: int = 2
    DROPOUT: float = 0.2
    USE_ITEM_BIAS: bool = True

    # training
    LOSS_TYPE: str = "bce"
    BATCH_SIZE: int = 512
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-6
    MAX_EPOCHS: int = 80
    PATIENCE: int = 8
    GRAD_CLIP: float = 5.0

    # evaluation
    K: int = 10
    EVAL_BATCH_SIZE: int = 512

    SEED: int = 42
    OUT_DIR: str = "/kaggle/working/sasrec_final"


CFG = Config()


In [3]:
# utils

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_dataset_path(domain: str):
    files = glob.glob("/kaggle/input/**/*.parquet", recursive=True)
    domain = domain.lower()

    if domain == "books":
        patterns = ["books_big.parquet", "books_big_nli.parquet", "books"]
    elif domain == "movies":
        patterns = ["movies_big.parquet", "movies_big_nli.parquet", "movies"]
    elif domain == "games":
        patterns = ["games_big.parquet", "games_big_nli.parquet", "games"]
    else:
        patterns = [domain]

    for pat in patterns:
        matches = [
            f for f in files
            if pat.lower() in os.path.basename(f).lower()
        ]
        if matches:
            return sorted(matches, key=lambda x: len(x))[0]

    print("Available parquet files:")
    for f in files:
        print(f)

    raise FileNotFoundError(
        f"Could not auto-detect dataset for domain={domain}. "
        "Set CFG.DATA_PATH manually."
    )


def ndcg_from_rank(rank_zero_based: int) -> float:
    return 1.0 / math.log2(rank_zero_based + 2.0)


def left_pad(seq, max_len):
    seq = seq[-max_len:]
    return [0] * (max_len - len(seq)) + seq


In [4]:
# load data

seed_everything(CFG.SEED)
os.makedirs(CFG.OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if CFG.DATA_PATH is None:
    CFG.DATA_PATH = find_dataset_path(CFG.DOMAIN)

print("DATA_PATH:", CFG.DATA_PATH)

df = pd.read_parquet(CFG.DATA_PATH)

required_cols = [CFG.USER_COL, CFG.ITEM_COL, CFG.TIME_COL]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Available columns: {df.columns.tolist()}"
    )

df = df[required_cols].copy()
df = df.dropna(subset=required_cols)

df[CFG.TIME_COL] = pd.to_numeric(df[CFG.TIME_COL], errors="coerce")
df = df.dropna(subset=[CFG.TIME_COL])

print("Rows:", len(df))
print("Users:", df[CFG.USER_COL].nunique())
print("Items:", df[CFG.ITEM_COL].nunique())

df = df.sort_values([CFG.USER_COL, CFG.TIME_COL, CFG.ITEM_COL]).reset_index(drop=True)

dup_count = df.duplicated([CFG.USER_COL, CFG.ITEM_COL]).sum()
print("Duplicate user-item rows before dedup:", int(dup_count))

if CFG.DEDUP_USER_ITEM:
    df = df.drop_duplicates(
        subset=[CFG.USER_COL, CFG.ITEM_COL],
        keep=CFG.DEDUP_KEEP
    ).reset_index(drop=True)

    print("\nAfter user-item deduplication:")
    print("Rows:", len(df))
    print("Users:", df[CFG.USER_COL].nunique())
    print("Items:", df[CFG.ITEM_COL].nunique())
    print("Duplicate user-item rows:", int(df.duplicated([CFG.USER_COL, CFG.ITEM_COL]).sum()))

df = df.sort_values([CFG.USER_COL, CFG.TIME_COL, CFG.ITEM_COL]).reset_index(drop=True)

user_values = df[CFG.USER_COL].unique()
item_values = df[CFG.ITEM_COL].unique()

user2idx = {u: idx for idx, u in enumerate(user_values)}
item2idx = {i: idx + 1 for idx, i in enumerate(item_values)}
idx2item = {idx + 1: i for idx, i in enumerate(item_values)}

df["user_idx"] = df[CFG.USER_COL].map(user2idx).astype(np.int64)
df["item_idx"] = df[CFG.ITEM_COL].map(item2idx).astype(np.int64)

num_users = len(user2idx)
num_items = len(item2idx)

print("\nMapped data:")
print("num_users:", num_users)
print("num_items:", num_items)


Device: cuda
DATA_PATH: /kaggle/input/datasets/rita12390/dataset/books_big.parquet

Loaded data:
Rows: 563929
Users: 32709
Items: 2000
Duplicate user-item rows before dedup: 11020

After user-item deduplication:
Rows: 552909
Users: 32709
Items: 2000
Duplicate user-item rows: 0

Mapped data:
num_users: 32709
num_items: 2000


In [5]:
# temprotal sequences

raw_sequences = {}

for u, g in df.groupby("user_idx", sort=False):
    seq = g.sort_values(CFG.TIME_COL)["item_idx"].tolist()

    if len(seq) >= CFG.MIN_USER_UNIQUE_ITEMS:
        raw_sequences[u] = seq

print("\nUsers after min unique sequence length filter:", len(raw_sequences))

train_sequences = {}
val_targets = {}
test_targets = {}

for u, seq in raw_sequences.items():
    train_seq = seq[:-2]
    val_item = seq[-2]
    test_item = seq[-1]

    if len(train_seq) >= 2:
        train_sequences[u] = train_seq
        val_targets[u] = val_item
        test_targets[u] = test_item

train_item_set = set()
for seq in train_sequences.values():
    train_item_set.update(seq)

filtered_users = []

for u in train_sequences.keys():
    if val_targets[u] in train_item_set and test_targets[u] in train_item_set:
        filtered_users.append(u)

train_sequences = {u: train_sequences[u] for u in filtered_users}
val_targets = {u: val_targets[u] for u in filtered_users}
test_targets = {u: test_targets[u] for u in filtered_users}

eval_users = list(train_sequences.keys())

all_user_items = {}

for u in eval_users:
    all_user_items[u] = set(train_sequences[u] + [val_targets[u], test_targets[u]])

val_seen_in_train = []
test_seen_in_train = []

for u in eval_users:
    train_set = set(train_sequences[u])
    val_seen_in_train.append(val_targets[u] in train_set)
    test_seen_in_train.append(test_targets[u] in train_set)

print("\nLeakage diagnostics:")
print("Val target already in train:", sum(val_seen_in_train), "/", len(val_seen_in_train))
print("Test target already in train:", sum(test_seen_in_train), "/", len(test_seen_in_train))

assert sum(val_seen_in_train) == 0, "Leakage: validation target appears in train history."
assert sum(test_seen_in_train) == 0, "Leakage: test target appears in train history."

train_lengths = np.array([len(s) for s in train_sequences.values()])

print("\nFinal sequential split:")
print("Evaluated users:", len(eval_users))
print("Candidate items from train:", len(train_item_set), "/", num_items)
print("Mean train length:", round(float(train_lengths.mean()), 2))
print("Median train length:", round(float(np.median(train_lengths)), 2))
print("Min / Max train length:", int(train_lengths.min()), int(train_lengths.max()))

candidate_items = sorted(train_item_set)
candidate_tensor = torch.LongTensor(candidate_items).to(device)
candidate_index = {item: idx for idx, item in enumerate(candidate_items)}




Users after min unique sequence length filter: 32663

Leakage diagnostics:
Val target already in train: 0 / 32661
Test target already in train: 0 / 32661

Final sequential split:
Evaluated users: 32661
Candidate items from train: 1999 / 2000
Mean train length: 14.92
Median train length: 11.0
Min / Max train length: 3 419


In [6]:
train_item_counts = np.zeros(num_items + 1, dtype=np.int64)

for seq in train_sequences.values():
    for item in seq:
        train_item_counts[item] += 1

pop_scores_all = torch.tensor(
    [train_item_counts[item] for item in candidate_items],
    dtype=torch.float32
)


def evaluate_popularity(context_mode: str):
    hits, ndcgs, recalls, precisions = [], [], [], []

    base_scores = pop_scores_all.clone()

    for u in eval_users:
        if context_mode == "val":
            context = train_sequences[u]
            target = val_targets[u]
        elif context_mode == "test_strict":
            context = train_sequences[u]
            target = test_targets[u]
        elif context_mode == "test_standard":
            context = train_sequences[u] + [val_targets[u]]
            target = test_targets[u]
        else:
            raise ValueError("Unknown context_mode")

        scores = base_scores.clone()

        for seen_item in set(context):
            if seen_item != target and seen_item in candidate_index:
                scores[candidate_index[seen_item]] = -1e9

        topk_local = torch.topk(scores, k=CFG.K).indices.numpy()
        topk_items = [candidate_items[i] for i in topk_local]

        if target in topk_items:
            rank = topk_items.index(target)
            hit = 1.0
            ndcg = ndcg_from_rank(rank)
        else:
            hit = 0.0
            ndcg = 0.0

        hits.append(hit)
        ndcgs.append(ndcg)
        recalls.append(hit)
        precisions.append(hit / CFG.K)

    return {
        "NDCG@10": float(np.mean(ndcgs)),
        "HR@10": float(np.mean(hits)),
        "Recall@10": float(np.mean(recalls)),
        "Precision@10": float(np.mean(precisions)),
        "n_users_evaluated": int(len(eval_users)),
    }


print("\nPopularity baseline:")
pop_val = evaluate_popularity("val")
pop_test_strict = evaluate_popularity("test_strict")
pop_test_standard = evaluate_popularity("test_standard")

print("VAL:", pop_val)
print("TEST strict:", pop_test_strict)
print("TEST standard:", pop_test_standard)



Popularity baseline:
VAL: {'NDCG@10': 0.0019550773032873683, 'HR@10': 0.004072134962187318, 'Recall@10': 0.004072134962187318, 'Precision@10': 0.00040721349621873187, 'n_users_evaluated': 32661}
TEST strict: {'NDCG@10': 0.0008475676326612304, 'HR@10': 0.0021432289274670095, 'Recall@10': 0.0021432289274670095, 'Precision@10': 0.00021432289274670097, 'n_users_evaluated': 32661}
TEST standard: {'NDCG@10': 0.0008481147211321163, 'HR@10': 0.0021432289274670095, 'Recall@10': 0.0021432289274670095, 'Precision@10': 0.00021432289274670097, 'n_users_evaluated': 32661}


In [7]:
# dataset

class SASRecTrainDataset(Dataset):
    def __init__(
        self,
        train_sequences,
        all_user_items,
        candidate_items,
        max_seq_len,
    ):
        self.users = list(train_sequences.keys())
        self.train_sequences = train_sequences
        self.all_user_items = all_user_items
        self.candidate_items = candidate_items
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.users)

    def sample_negative(self, user):
        seen = self.all_user_items[user]

        while True:
            x = random.choice(self.candidate_items)
            if x not in seen:
                return x

    def __getitem__(self, idx):
        user = self.users[idx]
        seq = self.train_sequences[user]

        # Shifted sequence:
        # input: i1, i2, ..., i(t-1)
        # pos:   i2, i3, ..., it
        tokens = seq[-(self.max_seq_len + 1):]

        input_items = tokens[:-1]
        pos_items = tokens[1:]

        input_items = input_items[-self.max_seq_len:]
        pos_items = pos_items[-self.max_seq_len:]

        pad_len = self.max_seq_len - len(input_items)

        input_items = [0] * pad_len + input_items
        pos_items = [0] * pad_len + pos_items

        neg_items = []

        for p in pos_items:
            if p == 0:
                neg_items.append(0)
            else:
                neg_items.append(self.sample_negative(user))

        return (
            torch.LongTensor(input_items),
            torch.LongTensor(pos_items),
            torch.LongTensor(neg_items),
        )


train_dataset = SASRecTrainDataset(
    train_sequences=train_sequences,
    all_user_items=all_user_items,
    candidate_items=candidate_items,
    max_seq_len=CFG.MAX_SEQ_LEN,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)


In [8]:
# model

class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        max_seq_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout,
        use_item_bias=True,
    ):
        super().__init__()

        self.num_items = num_items
        self.max_seq_len = max_seq_len
        self.hidden_dim = hidden_dim
        self.use_item_bias = use_item_bias

        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_dim)

        if self.use_item_bias:
            self.item_bias = nn.Embedding(num_items + 1, 1, padding_idx=0)
        else:
            self.item_bias = None

        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_emb.weight, mean=0.0, std=0.02)

        with torch.no_grad():
            self.item_emb.weight[0].fill_(0)

        if self.item_bias is not None:
            nn.init.zeros_(self.item_bias.weight)
            with torch.no_grad():
                self.item_bias.weight[0].fill_(0)

    def forward(self, seq):
        """
        seq: [B, L]
        returns: [B, L, H]
        """
        B, L = seq.shape

        positions = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)

        x = self.item_emb(seq) + self.pos_emb(positions)
        x = self.dropout(x)

        padding_mask = seq.eq(0)

        causal_mask = torch.triu(
            torch.ones(L, L, device=seq.device, dtype=torch.bool),
            diagonal=1,
        )

        x = self.encoder(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )

        x = self.layer_norm(x)
        return x

    def score_items(self, seq_repr, item_ids):
        """
        seq_repr: [B, L, H]
        item_ids: [B, L]
        returns: [B, L]
        """
        item_vec = self.item_emb(item_ids)
        scores = (seq_repr * item_vec).sum(dim=-1)

        if self.item_bias is not None:
            scores = scores + self.item_bias(item_ids).squeeze(-1)

        return scores

    def score_candidate_items_last(self, seq, candidate_tensor):
        """
        seq: [B, L]
        candidate_tensor: [C]
        returns: [B, C]
        """
        seq_repr = self.forward(seq)
        last_repr = seq_repr[:, -1, :]

        cand_vecs = self.item_emb(candidate_tensor)
        scores = last_repr @ cand_vecs.T

        if self.item_bias is not None:
            scores = scores + self.item_bias(candidate_tensor).squeeze(-1).unsqueeze(0)

        return scores


model = SASRec(
    num_items=num_items,
    max_seq_len=CFG.MAX_SEQ_LEN,
    hidden_dim=CFG.HIDDEN_DIM,
    num_heads=CFG.NUM_HEADS,
    num_layers=CFG.NUM_LAYERS,
    dropout=CFG.DROPOUT,
    use_item_bias=CFG.USE_ITEM_BIAS,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY,
)

bce_loss = nn.BCEWithLogitsLoss(reduction="none")

print("\nModel parameters:", sum(p.numel() for p in model.parameters()))



Model parameters: 233361


In [9]:
# train and eval

def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    total_positions = 0

    for seq, pos, neg in tqdm(loader, desc="train", leave=False):
        seq = seq.to(device)
        pos = pos.to(device)
        neg = neg.to(device)

        optimizer.zero_grad(set_to_none=True)

        seq_repr = model(seq)

        pos_scores = model.score_items(seq_repr, pos)
        neg_scores = model.score_items(seq_repr, neg)

        mask = pos.gt(0).float()

        if CFG.LOSS_TYPE.lower() == "bce":
            pos_loss = bce_loss(pos_scores, torch.ones_like(pos_scores))
            neg_loss = bce_loss(neg_scores, torch.zeros_like(neg_scores))
            loss_mat = pos_loss + neg_loss

        elif CFG.LOSS_TYPE.lower() == "bpr":
            loss_mat = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8)

        else:
            raise ValueError("CFG.LOSS_TYPE must be 'bce' or 'bpr'.")

        loss = (loss_mat * mask).sum() / mask.sum().clamp_min(1.0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()

        n = mask.sum().item()
        total_loss += loss.item() * n
        total_positions += n

    return total_loss / max(total_positions, 1)


@torch.no_grad()
def evaluate_sasrec(model, context_mode: str):
    model.eval()

    hits, ndcgs, recalls, precisions = [], [], [], []

    for start in tqdm(
        range(0, len(eval_users), CFG.EVAL_BATCH_SIZE),
        desc=f"eval_{context_mode}",
        leave=False,
    ):
        batch_users = eval_users[start:start + CFG.EVAL_BATCH_SIZE]

        batch_seq = []
        batch_targets = []
        batch_seen = []

        for u in batch_users:
            if context_mode == "val":
                context = train_sequences[u]
                target = val_targets[u]
                seen = set(train_sequences[u])

            elif context_mode == "test_strict":
                context = train_sequences[u]
                target = test_targets[u]
                seen = set(train_sequences[u])

            elif context_mode == "test_standard":
                context = train_sequences[u] + [val_targets[u]]
                target = test_targets[u]
                seen = set(train_sequences[u] + [val_targets[u]])

            else:
                raise ValueError("Unknown context_mode.")

            batch_seq.append(left_pad(context, CFG.MAX_SEQ_LEN))
            batch_targets.append(target)
            batch_seen.append(seen)

        seq_tensor = torch.LongTensor(batch_seq).to(device)
        scores = model.score_candidate_items_last(seq_tensor, candidate_tensor)

        for row_idx, (seen, target) in enumerate(zip(batch_seen, batch_targets)):
            for seen_item in seen:
                if seen_item != target and seen_item in candidate_index:
                    scores[row_idx, candidate_index[seen_item]] = -1e9

        topk_local = torch.topk(scores, k=CFG.K, dim=1).indices.cpu().numpy()

        for local_recs, target in zip(topk_local, batch_targets):
            rec_items = [candidate_items[i] for i in local_recs.tolist()]

            if target in rec_items:
                rank = rec_items.index(target)
                hit = 1.0
                ndcg = ndcg_from_rank(rank)
            else:
                hit = 0.0
                ndcg = 0.0

            hits.append(hit)
            ndcgs.append(ndcg)
            recalls.append(hit)
            precisions.append(hit / CFG.K)

    return {
        "NDCG@10": float(np.mean(ndcgs)),
        "HR@10": float(np.mean(hits)),
        "Recall@10": float(np.mean(recalls)),
        "Precision@10": float(np.mean(precisions)),
        "n_users_evaluated": int(len(eval_users)),
    }


In [10]:
# training

best_val_ndcg = -1.0
best_state = None
bad_epochs = 0
history = []


for epoch in range(1, CFG.MAX_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)

    val_metrics = evaluate_sasrec(model, "val")
    val_ndcg = val_metrics["NDCG@10"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"loss={train_loss:.4f} | "
        f"val_NDCG@10={val_metrics['NDCG@10']:.4f} | "
        f"val_HR@10={val_metrics['HR@10']:.4f}"
    )

    if val_ndcg > best_val_ndcg:
        best_val_ndcg = val_ndcg
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= CFG.PATIENCE:
        print(
            f"Early stopping at epoch {epoch}. "
            f"Best val NDCG@10={best_val_ndcg:.6f}"
        )
        break


if best_state is not None:
    model.load_state_dict(best_state)


sasrec_val = evaluate_sasrec(model, "val")
sasrec_test_strict = evaluate_sasrec(model, "test_strict")
sasrec_test_standard = evaluate_sasrec(model, "test_standard")


def print_metrics(title, metrics):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    for k, v in metrics.items():
        print(f"{k:20s} {v}")


print_metrics("SASRec — VAL", sasrec_val)
print_metrics("SASRec — TEST strict", sasrec_test_strict)
print_metrics("SASRec — TEST standard sequential", sasrec_test_standard)

print_metrics("Popularity — VAL", pop_val)
print_metrics("Popularity — TEST strict", pop_test_strict)
print_metrics("Popularity — TEST standard", pop_test_standard)

results = {
    "model": "SASRec-clean",
    "domain": CFG.DOMAIN,
    "data_path": CFG.DATA_PATH,
    "config": asdict(CFG),
    "num_users_mapped": num_users,
    "num_items_mapped": num_items,
    "num_candidate_items_train": len(candidate_items),
    "n_eval_users": len(eval_users),
    "popularity": {
        "val": pop_val,
        "test_strict": pop_test_strict,
        "test_standard": pop_test_standard,
    },
    "sasrec": {
        "val": sasrec_val,
        "test_strict": sasrec_test_strict,
        "test_standard": sasrec_test_standard,
    },
}

history_df = pd.DataFrame(history)

results_path = os.path.join(CFG.OUT_DIR, f"sasrec_clean_{CFG.DOMAIN}_results.json")
history_path = os.path.join(CFG.OUT_DIR, f"sasrec_clean_{CFG.DOMAIN}_history.csv")
model_path = os.path.join(CFG.OUT_DIR, f"sasrec_clean_{CFG.DOMAIN}_model.pt")

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

history_df.to_csv(history_path, index=False)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": asdict(CFG),
        "num_items": num_items,
        "candidate_items": candidate_items,
    },
    model_path,
)

print("\nSaved:")
print(results_path)
print(history_path)
print(model_path)


Starting SASRec training...


train:   0%|          | 0/64 [00:00<?, ?it/s]

eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 001 | loss=1.3009 | val_NDCG@10=0.0030 | val_HR@10=0.0058


train:   0%|          | 0/64 [00:00<?, ?it/s]

...

Epoch 034 | loss=0.6420 | val_NDCG@10=0.0049 | val_HR@10=0.0092
Early stopping at epoch 34. Best val NDCG@10=0.004894

Evaluating best SASRec model...


eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

eval_test_strict:   0%|          | 0/64 [00:00<?, ?it/s]

eval_test_standard:   0%|          | 0/64 [00:00<?, ?it/s]


SASRec — VAL
NDCG@10              0.004894346591028574
HR@10                0.009154649275894797
Recall@10            0.009154649275894797
Precision@10         0.0009154649275894799
n_users_evaluated    32661

SASRec — TEST strict
NDCG@10              0.0035283083911100055
HR@10                0.007654389026667891
Recall@10            0.007654389026667891
Precision@10         0.0007654389026667891
n_users_evaluated    32661

SASRec — TEST standard sequential
NDCG@10              0.004367443525227387
HR@10                0.008787238602614739
Recall@10            0.008787238602614739
Precision@10         0.000878723860261474
n_users_evaluated    32661

Popularity — VAL
NDCG@10              0.0019550773032873683
HR@10                0.004072134962187318
Recall@10            0.004072134962187318
Precision@10         0.00040721349621873187
n_users_evaluated    32661

Popularity — TEST strict
NDCG@10              0.0008475676326612304
HR@10                0.0021432289274670095
Recall@10    

# EASE + MultiVAE

In [11]:
import os
import json
import math
import glob
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [12]:
# config

@dataclass
class Config:
    DOMAIN: str = "books"
    DATA_PATH: str = None

    USER_COL: str = "user_id"
    ITEM_COL: str = "item_id"
    TIME_COL: str = "timestamp"

    DEDUP_USER_ITEM: bool = False
    UNMASK_TARGET_IF_SEEN: bool = True

    K: int = 10
    EVAL_BATCH_SIZE: int = 512

    EASE_LAMBDAS: tuple = (10.0, 25.0, 50.0, 100.0, 250.0, 500.0, 1000.0, 2000.0)

    VAE_HIDDEN_DIM: int = 600
    VAE_LATENT_DIM: int = 200
    VAE_DROPOUT: float = 0.5
    VAE_BATCH_SIZE: int = 512
    VAE_LR: float = 1e-3
    VAE_WEIGHT_DECAY: float = 0.0
    VAE_MAX_EPOCHS: int = 100
    VAE_PATIENCE: int = 10
    VAE_ANNEAL_CAP: float = 0.2
    VAE_TOTAL_ANNEAL_STEPS: int = 200000

    SEED: int = 42
    OUT_DIR: str = "/kaggle/working/ease_multivae_thesis_protocol"


CFG = Config()


In [13]:
# utils

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_dataset_path(domain: str):
    files = glob.glob("/kaggle/input/**/*.parquet", recursive=True)

    if not files:
        raise FileNotFoundError("No parquet files found under /kaggle/input.")

    domain = domain.lower()

    if domain == "books":
        patterns = ["books_big.parquet", "books_big_nli.parquet", "books"]
    elif domain == "movies":
        patterns = ["movies_big.parquet", "movies_big_nli.parquet", "movies"]
    elif domain == "games":
        patterns = ["games_big.parquet", "games_big_nli.parquet", "games"]
    else:
        patterns = [domain]

    for pat in patterns:
        matches = [f for f in files if pat.lower() in os.path.basename(f).lower()]
        if matches:
            return sorted(matches, key=lambda x: len(x))[0]

    print("Available parquet files:")
    for f in files:
        print(f)

    raise FileNotFoundError("Could not auto-detect dataset path. Set CFG.DATA_PATH manually.")


def ndcg_from_rank(rank_zero_based):
    return 1.0 / math.log2(rank_zero_based + 2.0)


def metrics_from_topk(topk, target):
    if target in topk:
        rank = topk.index(target)
        hit = 1.0
        ndcg = ndcg_from_rank(rank)
    else:
        hit = 0.0
        ndcg = 0.0

    return hit, ndcg, hit, hit / CFG.K


def print_metrics(title, metrics):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    for k, v in metrics.items():
        print(f"{k:20s} {v}")

In [14]:
# load data

seed_everything(CFG.SEED)
os.makedirs(CFG.OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if CFG.DATA_PATH is None:
    CFG.DATA_PATH = find_dataset_path(CFG.DOMAIN)

print("DATA_PATH:", CFG.DATA_PATH)

df = pd.read_parquet(CFG.DATA_PATH)

required_cols = [CFG.USER_COL, CFG.ITEM_COL, CFG.TIME_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Available: {df.columns.tolist()}")

df = df[required_cols].copy()
df = df.dropna(subset=required_cols)
df[CFG.TIME_COL] = pd.to_numeric(df[CFG.TIME_COL], errors="coerce")
df = df.dropna(subset=[CFG.TIME_COL])

df = df.sort_values([CFG.USER_COL, CFG.TIME_COL, CFG.ITEM_COL]).reset_index(drop=True)

print("\nLoaded data:")
print("Rows:", len(df))
print("Users:", df[CFG.USER_COL].nunique())
print("Items:", df[CFG.ITEM_COL].nunique())
print("Duplicate user-item rows:", int(df.duplicated([CFG.USER_COL, CFG.ITEM_COL]).sum()))

if CFG.DEDUP_USER_ITEM:
    df = df.drop_duplicates([CFG.USER_COL, CFG.ITEM_COL], keep="first").reset_index(drop=True)
    print("\nAfter dedup:")
    print("Rows:", len(df))
    print("Users:", df[CFG.USER_COL].nunique())
    print("Items:", df[CFG.ITEM_COL].nunique())


Device: cuda
DATA_PATH: /kaggle/input/datasets/rita12390/dataset/books_big.parquet

Loaded data:
Rows: 563929
Users: 32709
Items: 2000
Duplicate user-item rows: 11020


In [15]:
# map ids

user_values = df[CFG.USER_COL].unique()
item_values = df[CFG.ITEM_COL].unique()

user2idx = {u: idx for idx, u in enumerate(user_values)}
item2idx = {i: idx for idx, i in enumerate(item_values)}
idx2item = {idx: i for idx, i in enumerate(item_values)}

df["user_idx"] = df[CFG.USER_COL].map(user2idx).astype(np.int64)
df["item_idx"] = df[CFG.ITEM_COL].map(item2idx).astype(np.int64)

num_users = len(user2idx)
num_items = len(item2idx)

print("\nMapped:")
print("num_users:", num_users)
print("num_items:", num_items)



Mapped:
num_users: 32709
num_items: 2000


In [16]:
train_sequences = {}
val_targets = {}
test_targets = {}

for u, g in df.groupby("user_idx", sort=False):
    seq = g.sort_values(CFG.TIME_COL)["item_idx"].tolist()

    if len(seq) >= 3:
        train_sequences[u] = seq[:-2]
        val_targets[u] = seq[-2]
        test_targets[u] = seq[-1]

eval_users = [u for u in train_sequences if len(train_sequences[u]) > 0]

train_sequences = {u: train_sequences[u] for u in eval_users}
val_targets = {u: val_targets[u] for u in eval_users}
test_targets = {u: test_targets[u] for u in eval_users}

n_eval_users = len(eval_users)
eval_user2row = {u: row for row, u in enumerate(eval_users)}
row2user = {row: u for u, row in eval_user2row.items()}

train_lengths = np.array([len(train_sequences[u]) for u in eval_users])

print("\nTemporal split:")
print("Evaluated users:", n_eval_users)
print("Mean train length:", round(float(train_lengths.mean()), 2))
print("Median train length:", round(float(np.median(train_lengths)), 2))
print("Min / Max train length:", int(train_lengths.min()), int(train_lengths.max()))

val_seen = []
test_seen = []
for u in eval_users:
    s = set(train_sequences[u])
    val_seen.append(val_targets[u] in s)
    test_seen.append(test_targets[u] in s)

print("\nRepeated target diagnostics:")
print("Val target already in train:", sum(val_seen), "/", len(val_seen), "=", np.mean(val_seen))
print("Test target already in train:", sum(test_seen), "/", len(test_seen), "=", np.mean(test_seen))



Temporal split:
Evaluated users: 32709
Mean train length: 15.24
Median train length: 11.0
Min / Max train length: 7 459

Repeated target diagnostics:
Val target already in train: 412 / 32709 = 0.012595921611788805
Test target already in train: 224 / 32709 = 0.006848268060778379


In [17]:
# build train matrix

rows, cols = [], []

for u in eval_users:
    row = eval_user2row[u]
    for item in train_sequences[u]:
        rows.append(row)
        cols.append(item)

data = np.ones(len(rows), dtype=np.float32)

X_train = sp.csr_matrix(
    (data, (rows, cols)),
    shape=(n_eval_users, num_items),
    dtype=np.float32
)

X_train.data[:] = 1.0

print("\nTrain matrix:")
print("Shape:", X_train.shape)
print("Non-zero:", X_train.nnz)


Train matrix:
Shape: (32709, 2000)
Non-zero: 488270


In [18]:
# common evaluation

def evaluate_score_batch_fn(score_batch_fn, split_name, batch_size=512, unmask_target_if_seen=True):
    hits, ndcgs, recalls, precisions = [], [], [], []

    for start in tqdm(range(0, n_eval_users, batch_size), desc=f"eval_{split_name}", leave=False):
        batch_rows = list(range(start, min(start + batch_size, n_eval_users)))
        scores = score_batch_fn(batch_rows).astype(np.float32)

        for local_idx, row in enumerate(batch_rows):
            u = row2user[row]
            target = val_targets[u] if split_name == "val" else test_targets[u]

            seen_items = train_sequences[u]

            if unmask_target_if_seen:
                mask_items = [it for it in seen_items if it != target]
            else:
                mask_items = seen_items

            scores[local_idx, mask_items] = -1e9

            topk = np.argpartition(-scores[local_idx], CFG.K - 1)[:CFG.K]
            topk = topk[np.argsort(-scores[local_idx, topk])].tolist()

            hit, ndcg, recall, precision = metrics_from_topk(topk, target)

            hits.append(hit)
            ndcgs.append(ndcg)
            recalls.append(recall)
            precisions.append(precision)

    return {
        "NDCG@10": float(np.mean(ndcgs)),
        "HR@10": float(np.mean(hits)),
        "Recall@10": float(np.mean(recalls)),
        "Precision@10": float(np.mean(precisions)),
        "n_users_evaluated": int(n_eval_users),
    }



In [19]:
item_pop = np.asarray(X_train.sum(axis=0)).ravel().astype(np.float32)

def pop_score_batch(batch_rows):
    return np.tile(item_pop[None, :], (len(batch_rows), 1)).copy()

pop_val = evaluate_score_batch_fn(pop_score_batch, "val", CFG.EVAL_BATCH_SIZE, CFG.UNMASK_TARGET_IF_SEEN)
pop_test = evaluate_score_batch_fn(pop_score_batch, "test", CFG.EVAL_BATCH_SIZE, CFG.UNMASK_TARGET_IF_SEEN)

print_metrics("Popularity — VAL", pop_val)
print_metrics("Popularity — TEST", pop_test)


eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

eval_test:   0%|          | 0/64 [00:00<?, ?it/s]


Popularity — VAL
NDCG@10              0.001965560088246481
HR@10                0.004096731786358495
Recall@10            0.004096731786358495
Precision@10         0.0004096731786358495
n_users_evaluated    32709

Popularity — TEST
NDCG@10              0.0008296735790378263
HR@10                0.002078938518450579
Recall@10            0.002078938518450579
Precision@10         0.00020789385184505796
n_users_evaluated    32709


In [20]:
# EASE

def fit_ease(X, reg_lambda):
    G = (X.T @ X).toarray().astype(np.float64)
    diag = np.diag_indices(G.shape[0])
    G[diag] += reg_lambda

    P = np.linalg.inv(G)
    B = -P / np.diag(P)
    B[diag] = 0.0

    return B.astype(np.float32)


best_ease_B = None
best_ease_lambda = None
best_ease_val = -1.0
ease_all = {}

for lam in CFG.EASE_LAMBDAS:
    print(f"\nFitting EASE lambda={lam}")
    B = fit_ease(X_train, lam)

    def ease_score_batch(batch_rows, B=B):
        return np.asarray(X_train[batch_rows] @ B, dtype=np.float32)

    val_metrics = evaluate_score_batch_fn(
        ease_score_batch,
        "val",
        CFG.EVAL_BATCH_SIZE,
        CFG.UNMASK_TARGET_IF_SEEN
    )

    ease_all[str(lam)] = {"val": val_metrics}
    print_metrics(f"EASE lambda={lam} — VAL", val_metrics)

    if val_metrics["NDCG@10"] > best_ease_val:
        best_ease_val = val_metrics["NDCG@10"]
        best_ease_lambda = lam
        best_ease_B = B

print("\nBest EASE lambda:", best_ease_lambda)

def best_ease_score_batch(batch_rows):
    return np.asarray(X_train[batch_rows] @ best_ease_B, dtype=np.float32)

ease_val = evaluate_score_batch_fn(best_ease_score_batch, "val", CFG.EVAL_BATCH_SIZE, CFG.UNMASK_TARGET_IF_SEEN)
ease_test = evaluate_score_batch_fn(best_ease_score_batch, "test", CFG.EVAL_BATCH_SIZE, CFG.UNMASK_TARGET_IF_SEEN)

print_metrics("EASE — VAL", ease_val)
print_metrics("EASE — TEST", ease_test)



Tuning EASE...

Fitting EASE lambda=10.0


eval_val:   0%|          | 0/64 [00:00<?, ?it/s]


EASE lambda=10.0 — VAL
NDCG@10              0.12558075284854037
HR@10                0.20190161729187686
Recall@10            0.20190161729187686
Precision@10         0.020190161729187684
n_users_evaluated    32709

Fitting EASE lambda=25.0


eval_val:   0%|          | 0/64 [00:00<?, ?it/s]


EASE lambda=25.0 — VAL
NDCG@10              0.12595500309281515
HR@10                0.20281879605001682
Recall@10            0.20281879605001682
Precision@10         0.02028187960500168
n_users_evaluated    32709

Fitting EASE lambda=50.0


...

eval_val:   0%|          | 0/64 [00:00<?, ?it/s]


EASE lambda=2000.0 — VAL
NDCG@10              0.10965761554544873
HR@10                0.17949188296799046
Recall@10            0.17949188296799046
Precision@10         0.01794918829679905
n_users_evaluated    32709

Best EASE lambda: 50.0


eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

eval_test:   0%|          | 0/64 [00:00<?, ?it/s]


EASE — VAL
NDCG@10              0.12601402687863683
HR@10                0.2034608211807148
Recall@10            0.2034608211807148
Precision@10         0.020346082118071478
n_users_evaluated    32709

EASE — TEST
NDCG@10              0.05845890106220257
HR@10                0.10984744259989605
Recall@10            0.10984744259989605
Precision@10         0.010984744259989604
n_users_evaluated    32709


In [21]:
# MULTIVAE

class UserVectorDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].toarray().squeeze(0).astype(np.float32)
        return torch.from_numpy(x), idx


class MultiVAE(nn.Module):
    def __init__(self, n_items, hidden_dim=600, latent_dim=200, dropout=0.5):
        super().__init__()
        self.encoder_fc = nn.Linear(n_items, hidden_dim)
        self.encoder_mu = nn.Linear(hidden_dim, latent_dim)
        self.encoder_logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.decoder_fc2 = nn.Linear(hidden_dim, n_items)

        self.drop = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.normal_(m.bias, std=0.001)

    def encode(self, x):
        x = F.normalize(x, p=2, dim=1)
        x = self.drop(x)
        h = torch.tanh(self.encoder_fc(x))
        mu = self.encoder_mu(h)
        logvar = self.encoder_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z):
        h = torch.tanh(self.decoder_fc1(z))
        return self.decoder_fc2(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


def vae_loss(logits, x, mu, logvar, beta):
    log_softmax = F.log_softmax(logits, dim=1)
    nll = -torch.sum(log_softmax * x, dim=1).mean()
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    return nll + beta * kl, nll, kl


vae_dataset = UserVectorDataset(X_train)
vae_loader = DataLoader(
    vae_dataset,
    batch_size=CFG.VAE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=False
)

vae = MultiVAE(
    n_items=num_items,
    hidden_dim=CFG.VAE_HIDDEN_DIM,
    latent_dim=CFG.VAE_LATENT_DIM,
    dropout=CFG.VAE_DROPOUT
).to(device)

vae_optimizer = torch.optim.Adam(
    vae.parameters(),
    lr=CFG.VAE_LR,
    weight_decay=CFG.VAE_WEIGHT_DECAY
)

print("\nMultiVAE parameters:", sum(p.numel() for p in vae.parameters()))


def train_vae_epoch(model, loader, optimizer, update_count):
    model.train()

    total_loss, total_nll, total_kl, total_n = 0.0, 0.0, 0.0, 0

    for x, _ in tqdm(loader, desc="train_multivae", leave=False):
        x = x.to(device)

        if CFG.VAE_TOTAL_ANNEAL_STEPS > 0:
            beta = min(CFG.VAE_ANNEAL_CAP, update_count / CFG.VAE_TOTAL_ANNEAL_STEPS)
        else:
            beta = CFG.VAE_ANNEAL_CAP

        optimizer.zero_grad(set_to_none=True)

        logits, mu, logvar = model(x)
        loss, nll, kl = vae_loss(logits, x, mu, logvar, beta)

        loss.backward()
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_nll += nll.item() * bs
        total_kl += kl.item() * bs
        total_n += bs
        update_count += 1

    return {
        "loss": total_loss / total_n,
        "nll": total_nll / total_n,
        "kl": total_kl / total_n,
        "update_count": update_count,
    }


@torch.no_grad()
def vae_score_batch(batch_rows):
    vae.eval()

    X_batch = X_train[batch_rows].toarray().astype(np.float32)
    x = torch.from_numpy(X_batch).to(device)

    logits, _, _ = vae(x)
    return logits.detach().cpu().numpy().astype(np.float32)


best_vae_state = None
best_vae_val = -1.0
bad_epochs = 0
update_count = 0
vae_history = []

print("\nTraining MultiVAE...")

for epoch in range(1, CFG.VAE_MAX_EPOCHS + 1):
    train_stats = train_vae_epoch(vae, vae_loader, vae_optimizer, update_count)
    update_count = train_stats["update_count"]

    val_metrics = evaluate_score_batch_fn(
        vae_score_batch,
        "val",
        CFG.EVAL_BATCH_SIZE,
        CFG.UNMASK_TARGET_IF_SEEN
    )

    row = {
        "epoch": epoch,
        "train_loss": train_stats["loss"],
        "train_nll": train_stats["nll"],
        "train_kl": train_stats["kl"],
        "update_count": update_count,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    vae_history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"loss={train_stats['loss']:.4f} | "
        f"val_NDCG@10={val_metrics['NDCG@10']:.4f} | "
        f"val_HR@10={val_metrics['HR@10']:.4f}"
    )

    if val_metrics["NDCG@10"] > best_vae_val:
        best_vae_val = val_metrics["NDCG@10"]
        best_vae_state = {k: v.detach().cpu().clone() for k, v in vae.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= CFG.VAE_PATIENCE:
        print(f"Early stopping MultiVAE at epoch {epoch}. Best val NDCG@10={best_vae_val:.6f}")
        break


if best_vae_state is not None:
    vae.load_state_dict(best_vae_state)

multivae_val = evaluate_score_batch_fn(
    vae_score_batch,
    "val",
    CFG.EVAL_BATCH_SIZE,
    CFG.UNMASK_TARGET_IF_SEEN
)

multivae_test = evaluate_score_batch_fn(
    vae_score_batch,
    "test",
    CFG.EVAL_BATCH_SIZE,
    CFG.UNMASK_TARGET_IF_SEEN
)

print_metrics("MultiVAE — VAL", multivae_val)
print_metrics("MultiVAE — TEST", multivae_test)




MultiVAE parameters: 2763600

Training MultiVAE...


train_multivae:   0%|          | 0/64 [00:00<?, ?it/s]

eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 001 | loss=108.3519 | val_NDCG@10=0.0194 | val_HR@10=0.0415


train_multivae:   0%|          | 0/64 [00:00<?, ?it/s]

...

eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 061 | loss=74.4117 | val_NDCG@10=0.0884 | val_HR@10=0.1503
Early stopping MultiVAE at epoch 61. Best val NDCG@10=0.091751


eval_val:   0%|          | 0/64 [00:00<?, ?it/s]

eval_test:   0%|          | 0/64 [00:00<?, ?it/s]


MultiVAE — VAL
NDCG@10              0.09175056118234938
HR@10                0.15619554251123544
Recall@10            0.15619554251123544
Precision@10         0.015619554251123545
n_users_evaluated    32709

MultiVAE — TEST
NDCG@10              0.04838404786467687
HR@10                0.0898529456724449
Recall@10            0.0898529456724449
Precision@10         0.008985294567244489
n_users_evaluated    32709


In [22]:
summary = pd.DataFrame([
    {
        "model": "Popularity",
        "val_NDCG@10": pop_val["NDCG@10"],
        "val_HR@10": pop_val["HR@10"],
        "test_NDCG@10": pop_test["NDCG@10"],
        "test_HR@10": pop_test["HR@10"],
    },
    {
        "model": f"EASE(lambda={best_ease_lambda})",
        "val_NDCG@10": ease_val["NDCG@10"],
        "val_HR@10": ease_val["HR@10"],
        "test_NDCG@10": ease_test["NDCG@10"],
        "test_HR@10": ease_test["HR@10"],
    },
    {
        "model": "MultiVAE",
        "val_NDCG@10": multivae_val["NDCG@10"],
        "val_HR@10": multivae_val["HR@10"],
        "test_NDCG@10": multivae_test["NDCG@10"],
        "test_HR@10": multivae_test["HR@10"],
    },
])


print(summary.to_string(index=False))
results = {
    "config": asdict(CFG),
    "data_path": CFG.DATA_PATH,
    "num_users": num_users,
    "num_items": num_items,
    "n_eval_users": n_eval_users,
    "repeated_target_diagnostics": {
        "val_seen_in_train_count": int(sum(val_seen)),
        "val_seen_in_train_rate": float(np.mean(val_seen)),
        "test_seen_in_train_count": int(sum(test_seen)),
        "test_seen_in_train_rate": float(np.mean(test_seen)),
    },
    "popularity": {"val": pop_val, "test": pop_test},
    "ease": {
        "best_lambda": best_ease_lambda,
        "all_lambdas": ease_all,
        "val": ease_val,
        "test": ease_test,
    },
    "multivae": {"val": multivae_val, "test": multivae_test},
}

results_path = os.path.join(CFG.OUT_DIR, f"ease_multivae_thesis_{CFG.DOMAIN}_results.json")
summary_path = os.path.join(CFG.OUT_DIR, f"ease_multivae_thesis_{CFG.DOMAIN}_summary.csv")
vae_history_path = os.path.join(CFG.OUT_DIR, f"multivae_thesis_{CFG.DOMAIN}_history.csv")
ease_model_path = os.path.join(CFG.OUT_DIR, f"ease_thesis_{CFG.DOMAIN}_B.npy")
vae_model_path = os.path.join(CFG.OUT_DIR, f"multivae_thesis_{CFG.DOMAIN}_model.pt")

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

summary.to_csv(summary_path, index=False)
pd.DataFrame(vae_history).to_csv(vae_history_path, index=False)
np.save(ease_model_path, best_ease_B)

torch.save(
    {
        "model_state_dict": vae.state_dict(),
        "config": asdict(CFG),
        "num_items": num_items,
    },
    vae_model_path,
)

print(results_path)
print(summary_path)
print(vae_history_path)
print(ease_model_path)
print(vae_model_path)


SUMMARY — THESIS PROTOCOL
            model  val_NDCG@10  val_HR@10  test_NDCG@10  test_HR@10
       Popularity     0.001966   0.004097      0.000830    0.002079
EASE(lambda=50.0)     0.126014   0.203461      0.058459    0.109847
         MultiVAE     0.091751   0.156196      0.048384    0.089853

Saved:
/kaggle/working/ease_multivae_thesis_protocol/ease_multivae_thesis_books_results.json
/kaggle/working/ease_multivae_thesis_protocol/ease_multivae_thesis_books_summary.csv
/kaggle/working/ease_multivae_thesis_protocol/multivae_thesis_books_history.csv
/kaggle/working/ease_multivae_thesis_protocol/ease_thesis_books_B.npy
/kaggle/working/ease_multivae_thesis_protocol/multivae_thesis_books_model.pt
